# Programa 3 · Gestor de pedidos y stock

Esta aplicación representa productos y pedidos mediante clases. Los objetos interactúan para verificar existencias, procesar una cola de pedidos y calcular la facturación.

**Estructuras que aparecen:** variables tipadas, listas, `for`, `while`, funciones, condicionales, clases, objetos y métodos.

## 1. Clases `Producto` y `Pedido`

Cada objeto `Producto` administra sus datos y su stock. Cada objeto `Pedido` guarda su estado y puede confirmarse o rechazarse.

In [ ]:
class Producto:
    def __init__(
        self,
        codigo: str,
        nombre: str,
        precio: float,
        stock: int,
    ) -> None:
        self.codigo: str = codigo
        self.nombre: str = nombre
        self.precio: float = precio
        self.stock: int = stock

    def tiene_stock(self, cantidad: int) -> bool:
        return cantidad > 0 and cantidad <= self.stock

    def descontar_stock(self, cantidad: int) -> bool:
        if not self.tiene_stock(cantidad):
            return False

        self.stock -= cantidad
        return True

    def reponer_stock(self, cantidad: int) -> None:
        if cantidad > 0:
            self.stock += cantidad


class Pedido:
    def __init__(
        self,
        numero: int,
        codigo_producto: str,
        cantidad: int,
    ) -> None:
        self.numero: int = numero
        self.codigo_producto: str = codigo_producto
        self.cantidad: int = cantidad
        self.estado: str = "Pendiente"
        self.total: float = 0.0

    def confirmar(self, producto: Producto) -> None:
        self.total = producto.precio * self.cantidad
        self.estado = "Confirmado"

    def rechazar(self, motivo: str) -> None:
        self.estado = f"Rechazado: {motivo}"
        self.total = 0.0

## 2. Funciones de búsqueda y procesamiento

Las funciones coordinan los objetos: buscan el producto solicitado y le piden al pedido que se confirme o se rechace.

In [ ]:
def buscar_producto(
    productos: list[Producto],
    codigo: str,
) -> Producto | None:
    for producto in productos:
        if producto.codigo == codigo:
            return producto

    return None


def procesar_pedido(
    pedido: Pedido,
    productos: list[Producto],
) -> None:
    producto: Producto | None = buscar_producto(
        productos,
        pedido.codigo_producto,
    )

    if producto is None:
        pedido.rechazar("producto inexistente")
        return

    if producto.descontar_stock(pedido.cantidad):
        pedido.confirmar(producto)
    else:
        pedido.rechazar("stock insuficiente")


def calcular_facturacion(pedidos: list[Pedido]) -> float:
    total: float = 0.0

    for pedido in pedidos:
        if pedido.estado == "Confirmado":
            total += pedido.total

    return total

## 3. Creación de los objetos

Cada llamada a `Producto(...)` o `Pedido(...)` crea un objeto diferente, con sus propios atributos.

In [ ]:
productos: list[Producto] = [
    Producto("L01", "Lámpara modular", 48_000.0, 8),
    Producto("M02", "Mesa auxiliar", 92_000.0, 3),
    Producto("P03", "Panel acústico", 36_500.0, 10),
]

pendientes: list[Pedido] = [
    Pedido(1, "L01", 2),
    Pedido(2, "M02", 4),
    Pedido(3, "P03", 3),
    Pedido(4, "X99", 1),
]

procesados: list[Pedido] = []

## 4. Bucle principal e informe

In [ ]:
while pendientes:
    pedido_actual: Pedido = pendientes.pop(0)
    procesar_pedido(pedido_actual, productos)
    procesados.append(pedido_actual)

print("PEDIDOS PROCESADOS")
print("-" * 68)

for pedido in procesados:
    print(
        f"#{pedido.numero:<3} {pedido.codigo_producto:<4} "
        f"x {pedido.cantidad:<3} {pedido.estado:<34} "
        f"${pedido.total:>10,.2f}"
    )

print("\nSTOCK RESULTANTE")
print("-" * 68)

for producto in productos:
    print(
        f"{producto.codigo} · {producto.nombre:<22} "
        f"{producto.stock:>3} unidades"
    )

facturacion: float = calcular_facturacion(procesados)
confirmados: int = 0

for pedido in procesados:
    if pedido.estado == "Confirmado":
        confirmados += 1

print("\nRESUMEN")
print(f"Pedidos confirmados: {confirmados}")
print(f"Facturación total: ${facturacion:,.2f}")

## Para expandir

1. Usá el método `reponer_stock()` y volvé a procesar los pedidos rechazados por falta de existencias.
2. Agregá descuentos por cantidad mediante un método de `Pedido`.
3. Informá el producto más vendido.
4. Reemplazá algunos objetos de prueba por objetos creados con datos ingresados mediante `input()`.